In [1]:
import torch
from torchvision.io import read_video
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import face_alignment
from face_alignment.utils import mesure_temps

In [2]:
vid0 = read_video("./presentateur.mp4", output_format="TCHW")[0][:30]
hhh, www = vid0.shape[2], vid0.shape[3]

vid0.to(dtype=torch.float32)
vid0 = torch.nn.functional.interpolate(vid0, (hhh//2, www//2), mode="bilinear")
vid0.to(dtype=torch.uint8)

vid = vid0[:10]
vid2 = vid0[10:20]
vid3 = vid0[20:]

fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, face_detector='sfd')

C:\venv\TorchStandard\Lib\site-packages\torchvision\io\_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
C:\venv\TorchStandard\Lib\site-packages\torchvision\io\video.py:199: UserWarning: The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.
  warnings.warn("The pts_unit 'pts' gives wrong results. Please use pts_unit 'sec'.")


In [3]:
# Face alignment accepts THWC
landmarks, _, bboxs = fa.get_landmarks_from_batch(vid, return_bboxes=True)
landmarks, _, bboxs = fa.get_landmarks_from_batch(vid2, return_bboxes=True)
landmarks, _, bboxs = fa.get_landmarks_from_batch(vid3, return_bboxes=True)

Face detector time: 8.331922054290771
Face detector time: 0.6689023971557617
Face detector time: 0.6046180725097656


In [4]:
from face_alignment.detection.retina.pytorch_retinaface import Pytorch_RetinaFace
rf = Pytorch_RetinaFace(top_k=50, keep_top_k=10, device="cuda", confidence_threshold=0.5)

Using device: cuda
Loading pretrained model from ./face_alignment/detection/retina/weights/mobilenet0.25_Final.pth
remove prefix 'module.'
Missing keys:0
Unused checkpoint keys:0
Used keys:300


In [5]:
# dets = rf.detect_faces(retina_input)
# cropped_faces, infos = rf.center_and_crop_rescale(retina_input, dets, scale_factor=1, shift_factor=0.5, aspect_ratio=1)
# infos

In [6]:
vid0 = vid0.permute(0, 2, 3, 1).to(dtype=torch.float32)

@mesure_temps
def process(video):
    for idx in range(video.shape[0]):
        dets = rf.detect_faces(video[idx])

process(vid0)

process éxecutée en 0.8617 secondes
